# 10 — Validate overall S. aureus first-culture selection

The overall branch selects the earliest *Staphylococcus aureus* culture per source release, cohort, and patient within the configured reporting window, irrespective of phenotype. Other reporting categories independently select their own first qualifying cultures.

This notebook checks overall *S. aureus* grain, earliest-culture selection, source-label consolidation, and AST reconciliation. The obsolete all-organism/full-release comparisons have been removed. The corrected ranking query has no saved result until rerun; other saved outputs retain their original execution scope.

Use Notebook 13 and the current dbt assertions for category-specific reporting. Model names containing `isolate` refer to analytical first cultures; ARMD does not supply verified biological isolate IDs.


## 1. Check the actual overall S. aureus table's grain

In the current model, each `source_version × cohort_id × patient_key × organism_reporting_group` refers to one selected overall *S. aureus* culture within the window. Keys must be unique and source-key arrays nonempty. A fresh execution should produce zero structural error counts. This query does not validate the independent MRSA/MSSA selections.


In [ ]:
%sql
with duplicate_groups as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        count(*) as n
    from workspace.micro_dev.int_overall_first_reporting_isolate
    group by
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group
    having count(*) > 1
)
select
    count(*) as overall_records,
    count(distinct overall_reporting_isolate_key) as unique_overall_keys,
    count(*) - count(distinct overall_reporting_isolate_key) as duplicate_keys,
    (select count(*) from duplicate_groups) as duplicate_groups,
    sum(case when source_culture_organism_keys is null
              or size(source_culture_organism_keys) = 0
             then 1 else 0 end) as records_missing_source_keys
from workspace.micro_dev.int_overall_first_reporting_isolate;


### Independently reconstruct the earliest eligible S. aureus culture

Rank the window-eligible observations in `int_culture_reporting_observation`, restricted to *S. aureus*, using timestamp then culture key. A full outer join detects missing, extra, or differently selected patient groups. Expect `selection_mismatches = 0`.

The observation model already applies the reporting window before ranking. This query checks selection from that eligible population; the dbt window assertions separately check date bounds. Its saved output was cleared after correcting the source scope; rerun it against the current build.


In [ ]:
%sql
with observed_cultures as (
    select distinct
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        culture_key,
        culture_time_shifted
    from workspace.micro_dev.int_culture_reporting_observation
    where culture_time_shifted is not null
      and organism_reporting_group = 'Staphylococcus aureus'
),
independent_rank as (
    select
        *,
        row_number() over (
            partition by
                source_version,
                cohort_id,
                patient_key,
                organism_reporting_group
            order by culture_time_shifted, culture_key
        ) as rn
    from observed_cultures
),
expected as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        culture_key
    from independent_rank
    where rn = 1
),
actual as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        culture_key
    from workspace.micro_dev.int_overall_first_reporting_isolate
),
comparison as (
    select
        e.culture_key as expected_culture,
        a.culture_key as selected_culture
    from expected e
    full outer join actual a
        on e.source_version = a.source_version
       and e.cohort_id = a.cohort_id
       and e.patient_key = a.patient_key
       and e.organism_reporting_group = a.organism_reporting_group
)
select
    count(*) as compared_groups,
    sum(case when expected_culture = selected_culture
             then 0 else 1 end) as selection_mismatches
from comparison;


### Check source-label consolidation on selected S. aureus cultures

This query begins with the actual selected overall table and rejoins source labels on the same culture. It therefore checks source-key consolidation for the selected *S. aureus* records. Both mismatch counts should be zero on a fresh execution. It cannot prove which culture should have been selected or validate phenotype-specific AST scopes.

In [ ]:
%sql
with expected_source_records as (
    select
        f.overall_reporting_isolate_key,
        array_sort(array_distinct(flatten(
            collect_list(r.source_culture_organism_keys)
        ))) as expected_source_keys,
        count(distinct r.reporting_culture_organism_key)
            as expected_phenotype_records_on_culture
    from workspace.micro_dev.int_overall_first_reporting_isolate f
    inner join workspace.micro_dev.int_reporting_culture_organism r
        on f.source_version = r.source_version
       and f.cohort_id = r.cohort_id
       and f.patient_key = r.patient_key
       and f.organism_reporting_group = r.organism_reporting_group
       and f.culture_key = r.culture_key
    group by f.overall_reporting_isolate_key
)
select
    count(*) as selected_cultures_compared,
    sum(case
        when e.overall_reporting_isolate_key is null
          or f.source_culture_organism_keys <> e.expected_source_keys
            then 1 else 0
    end) as source_key_array_mismatches,
    sum(case
        when e.overall_reporting_isolate_key is null
          or f.phenotype_records_on_culture
             <> e.expected_phenotype_records_on_culture
            then 1 else 0
    end) as phenotype_record_count_mismatches
from workspace.micro_dev.int_overall_first_reporting_isolate f
left join expected_source_records e
    on f.overall_reporting_isolate_key = e.overall_reporting_isolate_key;


## 2. Check overall S. aureus AST reconciliation

`int_overall_first_reporting_susceptibility` now attaches the `overall` scope of the shared `int_culture_reporting_susceptibility` model to selected *S. aureus* cultures. That scope reconciles all retained source labels on the same culture. Stratified AST is handled separately and is not mixed into this check.

Keys must be unique. Resolved endpoints must contain an interpretation, and unresolved endpoints must not. Saved outputs are from their original execution; rerun the notebook to record the current build.


In [ ]:
%sql
select
    count(*) as endpoint_rows,
    count(distinct overall_reporting_susceptibility_key) as unique_keys,
    sum(case when resolution_status = 'resolved'
              and susceptibility is null
             then 1 else 0 end) as resolved_with_null,
    sum(case when resolution_status <> 'resolved'
              and susceptibility is not null
             then 1 else 0 end) as unresolved_with_result
from workspace.micro_dev.int_overall_first_reporting_susceptibility;


### Inspect overall AST resolution reasons

The output describes resolved, missing, and conflicting endpoints on selected overall *S. aureus* cultures, including multi-source-label reconciliation. These are AST endpoint counts, not additional patient denominators. The query is an audit of the currently built table, not a criterion for selecting a later tested culture.

In [ ]:
%sql
select
    resolution_status,
    reconciliation_reason,
    count(*) as endpoint_rows,
    sum(case when source_organisms_with_endpoint > 1
             then 1 else 0 end) as multi_source_endpoints
from workspace.micro_dev.int_overall_first_reporting_susceptibility
group by resolution_status, reconciliation_reason
order by resolution_status, reconciliation_reason;


## Conclusion and validation route

Run these checks against consistently built models in the same reporting window. The overall branch covers *S. aureus* only; it does not define the independently selected base, ESCE-R, VRE, MRSA, or MSSA populations.

`assert_first_stratified_qualifying_culture` checks category-specific selection, `assert_reporting_window_selection` checks window bounds, and the denominator/completeness assertions cover the reporting grid. Notebook 13 records the current design and saved 2023 coverage; Notebook 14 explains the shifted-date period choice. No warehouse execution was performed during this cleanup.
